# How to solve a scheduling problem with Z3

**Task: Assign peer reviewers to group presentations**

Imagine you're teaching a course with 21 project groups. You have seven presentation slots on each of three days, and you want the groups to review one another's presentations. You've already assigned the presentation slots. Your task is to decide who reviews each talk.

| Day | Presentation order |
| --- | --- |
| 1 | Group1 through Group7 |
| 2 | Group8 through Group14 |
| 3 | Group15 through Group21 |

1. Every group gives exactly **two reviews each day**, even when it presents.
2. Every talk receives exactly **six reviews** from different groups. That gives us 21 × 2 = 42 reviews to split evenly among seven talks.
3. On its presentation day, a group skips its own talk and the talks immediately before and after it.
4. No group reviews two consecutive talks on the same day.

Can we solve Day 1 without knowing anyone's assignments on Day 2? Yes. Every rule starts over each day, so we'll work through one day first and then repeat for the other two.

All the code is in the cells below. The backtracking section uses the standard library. Install **requirements.txt** in your notebook's Python environment before running the Z3 section.

In [1]:
from functools import cache
from itertools import combinations

GROUPS = [f"Group{i}" for i in range(1, 22)]
PRESENTATION_DAYS = [GROUPS[start:start + 7] for start in (0, 7, 14)]
REVIEWS_PER_GROUP = 2
REVIEWS_PER_TALK = 6

## 1. Start with choices we could make by hand

On Day 1, Group4 can review talks 1, 2, 6, and 7 after we exclude its own talk and its neighbors. Which pairs can it actually take? Pairs (1, 2) and (6, 7) are consecutive, so the four legal pairs are (1, 6), (1, 7), (2, 6), and (2, 7).

Make a list of legal pairs for each group and give each talk six review slots. Pick a pair for the group with the fewest choices, subtract one slot from each of its talks, then move on. If we get stuck, go back and try a different pair. A partial assignment only matters to the rest of the search through the slots it leaves open, so we can remember search results by the next group and the seven remaining slot counts.

In [2]:
def solve_with_backtracking(presenters, reviewers, forbidden_reviews=()):
    """Assign review pairs, optionally excluding (reviewer, presenter) pairs."""
    forbidden = set(forbidden_reviews)
    positions = {talk: i for i, talk in enumerate(presenters)}
    choices = {}
    for group in reviewers:
        own_talk = positions.get(group)
        eligible = [
            i for i in range(len(presenters))
            if (group, presenters[i]) not in forbidden
            and (own_talk is None or abs(i - own_talk) > 1)
        ]
        choices[group] = [(a, b) for a, b in combinations(eligible, 2) if b - a > 1]

    # Try the groups with the fewest choices first.
    ordered_groups = sorted(reviewers, key=lambda group: len(choices[group]))

    @cache
    def search(group_index, remaining):
        if group_index == len(ordered_groups):
            return () if not any(remaining) else None

        group = ordered_groups[group_index]
        # Prefer talks that still need the most reviews.
        pairs = sorted(choices[group], key=lambda pair: -sum(remaining[i] for i in pair))
        for a, b in pairs:
            if remaining[a] == 0 or remaining[b] == 0:
                continue
            next_remaining = list(remaining)
            next_remaining[a] -= 1
            next_remaining[b] -= 1
            rest = search(group_index + 1, tuple(next_remaining))
            if rest is not None:
                return ((a, b),) + rest
        return None

    pairs = search(0, (REVIEWS_PER_TALK,) * len(presenters))
    if pairs is None:
        raise RuntimeError("the constraints have no solution")

    assignments = {talk: [] for talk in presenters}
    for group, pair in zip(ordered_groups, pairs):
        for i in pair:
            assignments[presenters[i]].append(group)
    return assignments

In [3]:
presenters = PRESENTATION_DAYS[0]
group = "Group4"
position = presenters.index(group)
eligible = [i for i in range(len(presenters)) if abs(i - position) > 1]
legal_pairs = [
    (presenters[a], presenters[b])
    for a, b in combinations(eligible, 2)
    if b - a > 1
]
print(f"{group}'s legal pairs on Day 1: {legal_pairs}")
print("Starting review slots per talk:", [REVIEWS_PER_TALK] * len(presenters))

backtracking_schedules = {
    day: solve_with_backtracking(talks, GROUPS)
    for day, talks in enumerate(PRESENTATION_DAYS, start=1)
}
print("Backtracking, Group1's reviewers:", ", ".join(backtracking_schedules[1]["Group1"]))

Group4's legal pairs on Day 1: [('Group1', 'Group6'), ('Group1', 'Group7'), ('Group2', 'Group6'), ('Group2', 'Group7')]
Starting review slots per talk: [6, 6, 6, 6, 6, 6, 6]
Backtracking, Group1's reviewers: Group6, Group5, Group8, Group11, Group14, Group18


### Before we model the schedule: SAT, SMT, and Z3

A **constraint** is a rule that a solution must satisfy. A set of constraints is **satisfiable** if there is some choice of values that makes every rule true at the same time.

**SAT** (Boolean satisfiability) asks whether true or false values can make a Boolean formula true. Suppose two variables say whether a group reviews Talk A and Talk B. The constraints “review at least one” and “do not review both” can hold together. Reviewing A alone is one solution.

**SMT** (satisfiability modulo theories) extends SAT by allowing formulas to also use rules from theories such as integer arithmetic. For example, you can ask whether two integer variables can add to six while each is at least four. They cannot, so those constraints are unsatisfiable.

**Z3** is a solver that checks constraints for us. We declare variables, add rules, and ask whether they can all hold. Z3 returns `sat` when it finds a solution, `unsat` when it proves there is none, or `unknown` when it cannot decide. After `sat`, we can ask for a *model*, which gives values for our variables. For the review schedule, the values we need are Boolean decisions, and Z3 lets us express the required counts directly.

## 2. Decide what a variable should mean

Backtracking made me spell out how to search. For Z3, I start with a different question: what are the smallest decisions that describe the finished schedule? For any talk and any reviewing group, there's a yes-or-no decision: **does this group review this talk?**

For one day, let $p_i$ be the group presenting the $i$-th talk and $g_j$ be the $j$-th group in the class list. Let $x_{ij}$ be true if $g_j$ reviews the talk by $p_i$. There are **7 rows** because only seven groups present today, and **21 columns** because all 21 groups owe reviews today. We keep cells for forbidden pairs and constrain them to false.

Here are the first three rows and a few selected columns from Day 1. An open cell is marked **?**, and **0** marks an assignment we already know is forbidden by the presenter rule.

| Talk ↓ / Reviewer → | Group1 | Group2 | Group3 | Group4 | Group8 | … |
| --- | --- | --- | --- | --- | --- | --- |
| 1 · Group1 | 0 | 0 | ? | ? | ? | … |
| 2 · Group2 | 0 | 0 | 0 | ? | ? | … |
| 3 · Group3 | ? | 0 | 0 | 0 | ? | … |

For instance, the cell in row 2 and the Group3 column is forbidden because Group3 presents immediately after Group2. On Day 2, the first row is Group8, while the reviewer columns still start at Group1. That is why the code checks group names to identify presenters.

## 3. Translate each rule into a constraint

To write the counts, let $[x_{ij}]$ equal 1 when $x_{ij}$ is true and 0 when it is false. Think of each sum as running a finger along one row or column and counting the yeses.

| Scheduling rule | Formula | How to read it |
| --- | --- | --- |
| Two reviews per group | $\sum_{i=1}^{7} [x_{ij}] = 2 \quad (j=1,\ldots,21)$ | Count down each reviewer column. |
| Six reviewers per talk | $\sum_{j=1}^{21} [x_{ij}] = 6 \quad (i=1,\ldots,7)$ | Count across each talk row. |
| No self or neighboring reviews | $\lnot x_{ij}$ whenever $g_j=p_k$ and $\lvert i-k\rvert\leq 1$ | If the reviewer presents at position $k$, rule out talks within one position of it. |
| No consecutive reviews | $\lnot x_{ij}\lor\lnot x_{i+1,j} \quad (i=1,\ldots,6;\ j=1,\ldots,21)$ | At least one of two adjacent cells in each reviewer column must be false. |

Try the last formula with both cells set to true. Both negations become false, so the whole constraint fails. This rule applies to every column, including groups that do not present that day.

In the code, `PbEq` expresses each counting constraint directly. Each true Boolean contributes its weight of 1. `Not` blocks individual cells, and `Or(Not(a), Not(b))` blocks pairs.

The daily totals agree because $21 \times 2 = 7 \times 6$. We still need the column and row constraints separately. One balances the work across reviewers, and the other balances the feedback across talks.

In [4]:
def build_problem(presenters, reviewers):
    """Return a Z3 solver and a matrix indexed by talk, then reviewer."""
    from z3 import Bool, Not, Or, PbEq, Solver

    review = [
        [Bool(f"review_{talk}_{group}") for group in reviewers]
        for talk in presenters
    ]
    solver = Solver()

    # Rule 1. Each reviewer column has exactly two true cells.
    for column in zip(*review):
        solver.add(PbEq([(entry, 1) for entry in column], REVIEWS_PER_GROUP))

    # Rule 2. Each talk row has exactly six true cells.
    for row in review:
        solver.add(PbEq([(entry, 1) for entry in row], REVIEWS_PER_TALK))

    # Rule 3. Block the presenting group and its neighbors as reviewers.
    for i, row in enumerate(review):
        blocked = presenters[max(0, i - 1):i + 2]
        for group, entry in zip(reviewers, row):
            if group in blocked:
                solver.add(Not(entry))

    # Rule 4. One reviewer cannot take both talks in an adjacent pair.
    for earlier_row, later_row in zip(review, review[1:]):
        for earlier_review, later_review in zip(earlier_row, later_row):
            solver.add(Or(Not(earlier_review), Not(later_review)))

    return solver, review

## 4. Ask Z3 for a schedule

A result of **sat** means the constraints can all hold together. The *model* gives the values that make that happen, so we can collect the true cells in each row to recover the list of reviewers. A result of **unsat** means no such assignment exists.

Below is the full matrix for one Day 1 solution. The rows show presenting groups, and the columns show reviewer group numbers. A 1 means that reviewer is assigned to the talk. Read across a row to find its six reviewers, or down a column to find a group's two talks.

In [5]:
def read_assignments(model, review, presenters, reviewers):
    """Convert the true cells in a satisfying model to reviewer names."""
    from z3 import is_true

    return {
        talk: [
            group
            for group, entry in zip(reviewers, row)
            if is_true(model.eval(entry))
        ]
        for talk, row in zip(presenters, review)
    }


def solve_with_z3(presenters, reviewers):
    """Build the constraints, ask Z3 for a model, and decode it."""
    from z3 import sat, unsat

    solver, review = build_problem(presenters, reviewers)
    status = solver.check()
    if status == unsat:
        raise RuntimeError("the constraints have no solution")
    if status != sat:
        raise RuntimeError(f"Z3 returned {status}: {solver.reason_unknown()}")
    return read_assignments(solver.model(), review, presenters, reviewers)

In [6]:
from z3 import is_true, sat

solver, review = build_problem(presenters, GROUPS)
print(f"Day 1 model: {len(review)} talk rows × {len(review[0])} reviewer columns")
status = solver.check()
print("Z3 result:", status)
if status != sat:
    raise RuntimeError(f"Expected a model for Day 1; Z3 returned {status}")

model = solver.model()
print("Day 1 assignment matrix (columns are reviewer group numbers):")
print(f"{'Talk':<8}" + "".join(f"{j:>3}" for j in range(1, len(GROUPS) + 1)))
for talk, row in zip(presenters, review):
    values = [int(is_true(model.eval(entry))) for entry in row]
    print(f"{talk:<8}" + "".join(f"{value:>3}" for value in values))

z3_schedules = {1: read_assignments(model, review, presenters, GROUPS)}
for day, talks in enumerate(PRESENTATION_DAYS[1:], start=2):
    z3_schedules[day] = solve_with_z3(talks, GROUPS)
print("Z3, Group1's reviewers:", ", ".join(z3_schedules[1]["Group1"]))

Day 1 model: 7 talk rows × 21 reviewer columns
Z3 result: sat
Day 1 assignment matrix (columns are reviewer group numbers):
Talk      1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21
Group1    0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1  1  1
Group2    0  0  0  1  1  1  1  1  0  0  0  0  0  1  0  0  0  0  0  0  0
Group3    0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  1  1  1  1  1
Group4    0  1  0  0  0  1  1  0  0  1  0  1  1  0  0  0  0  0  0  0  0
Group5    1  0  1  0  0  0  0  1  1  0  1  0  0  0  1  0  0  0  0  0  0
Group6    0  1  0  1  0  0  0  0  0  1  0  1  1  1  0  0  0  0  0  0  0
Group7    1  0  1  0  1  0  0  0  1  0  1  0  0  0  1  0  0  0  0  0  0
Z3, Group1's reviewers: Group16, Group17, Group18, Group19, Group20, Group21


## 5. Check the schedules against the original rules

Backtracking and Z3 can choose different reviewers. To check either answer, go back to the four rules and apply them to the actual schedules. For the consecutive-talk rule, compare the reviewer lists of each adjacent pair of talks and check that they have no group in common.

In [7]:
from collections import Counter

for method, schedules in [("Backtracking", backtracking_schedules), ("Z3", z3_schedules)]:
    for day, talks in enumerate(PRESENTATION_DAYS, start=1):
        schedule = schedules[day]
        assert list(schedule) == talks
        counts = Counter(group for reviewers in schedule.values() for group in reviewers)
        assert counts == {group: 2 for group in GROUPS}
        for i, talk in enumerate(talks):
            reviewers = set(schedule[talk])
            assert len(schedule[talk]) == len(reviewers) == 6
            assert reviewers.isdisjoint(talks[max(0, i - 1):i + 2])
            if i:
                assert reviewers.isdisjoint(schedule[talks[i - 1]])
        print(f"{method}, Day {day}: {sum(counts.values())} reviews across {len(talks)} talks")

Backtracking, Day 1: 42 reviews across 7 talks
Backtracking, Day 2: 42 reviews across 7 talks
Backtracking, Day 3: 42 reviews across 7 talks
Z3, Day 1: 42 reviews across 7 talks
Z3, Day 2: 42 reviews across 7 talks
Z3, Day 3: 42 reviews across 7 talks


## 6. Compare how long the two methods take

Each timed run solves all three days from scratch and returns the schedules. This includes building the legal pairs or Z3 constraints, searching, and converting the results to reviewer names. The imports have already run, and printing is outside the timer.

We time five batches of 20 runs for each method. The reported number is the median batch time divided by 20, in milliseconds per three-day schedule. These times describe this instance on this machine. Larger or tighter schedules can change how much work each method has to do.

In [8]:
from statistics import median
from timeit import repeat


def solve_all_days(solve):
    return [solve(talks, GROUPS) for talks in PRESENTATION_DAYS]


runs_per_batch = 20
batches = 5
for name, solve in [("Backtracking", solve_with_backtracking), ("Z3", solve_with_z3)]:
    batch_times = repeat(lambda: solve_all_days(solve), repeat=batches, number=runs_per_batch)
    milliseconds = 1000 * median(batch_times) / runs_per_batch
    print(f"{name:<12} {milliseconds:8.3f} ms per three-day schedule")

Backtracking    0.192 ms per three-day schedule


Z3             29.770 ms per three-day schedule


## 7. Prevent reciprocal reviews across the whole schedule

**New rule.** For every pair of distinct groups, at most one group may review the other's presentation. This applies whether the groups present on the same day or on different days. For example, if Group1 reviews Group4 on Day 1, Group4 cannot review Group1 that day. If Group16 reviews Group1 on Day 1, Group1 cannot review Group16 on Day 3. Either direction is allowed when it also satisfies the original rules, and both directions may be absent.

**With backtracking**, we still precompute legal pairs of talks for each reviewer on each day. We now search over all three days together. Before assigning a review, we check whether its reverse is already in the partial schedule. If a choice leads to a dead end, we undo both reviews in that choice and try another pair. This lets a later choice send the search back to an earlier day.

**With Z3**, we put the existing daily constraints into one solver. For each of the 210 unordered pairs of groups, we add a clause that says at least one review direction must be false. We use the same clause for same-day and cross-day pairs. The solver decides which directions to use while satisfying the other rules.

In [9]:
def solve_extended_with_backtracking():
    """Assign one pair of talks per reviewer per day, with no reciprocal reviews."""
    positions = {
        talk: i for talks in PRESENTATION_DAYS for i, talk in enumerate(talks)
    }
    slots = dict.fromkeys(GROUPS, REVIEWS_PER_TALK)
    reviewed = {group: set() for group in GROUPS}
    assignments = {talk: [] for talk in GROUPS}

    # Precompute the pairs that satisfy the rules about presentation order.
    pending = []
    for talks in PRESENTATION_DAYS:
        for reviewer in GROUPS:
            eligible = [
                talk for talk in talks
                if reviewer not in talks or abs(positions[talk] - positions[reviewer]) > 1
            ]
            pairs = [
                (a, b) for a, b in combinations(eligible, 2)
                if positions[b] - positions[a] > 1
            ]
            pending.append((reviewer, pairs))

    def search(remaining):
        if not remaining:
            return not any(slots.values())

        # Recheck which pairs fit the slots and the reviews already assigned.
        best_index = None
        best_pairs = None
        for i, (reviewer, pairs) in enumerate(remaining):
            viable = [
                (a, b) for a, b in pairs
                if slots[a] and slots[b]
                and reviewer not in reviewed[a] and reviewer not in reviewed[b]
            ]
            if not viable:
                return False
            if best_pairs is None or len(viable) < len(best_pairs):
                best_index, best_pairs = i, viable

        reviewer, _ = remaining[best_index]
        rest = remaining[:best_index] + remaining[best_index + 1:]
        for a, b in sorted(best_pairs, key=lambda pair: -(slots[pair[0]] + slots[pair[1]])):
            for talk in (a, b):
                slots[talk] -= 1
                reviewed[reviewer].add(talk)
                assignments[talk].append(reviewer)
            if search(rest):
                return True
            for talk in (a, b):
                slots[talk] += 1
                reviewed[reviewer].remove(talk)
                assignments[talk].pop()
        return False

    if not search(pending):
        raise RuntimeError("No schedule satisfies the no-reciprocal-review rule.")
    return {
        day: {talk: assignments[talk] for talk in talks}
        for day, talks in enumerate(PRESENTATION_DAYS, start=1)
    }

extended_backtracking_schedules = solve_extended_with_backtracking()
print("Backtracking found a schedule with no reciprocal reviews.")

Backtracking found a schedule with no reciprocal reviews.


In [10]:
from z3 import Not, Or, Solver

joint_solver = Solver()
daily_models = []
review_for = {}
for talks in PRESENTATION_DAYS:
    daily_solver, daily_review = build_problem(talks, GROUPS)
    joint_solver.add(*daily_solver.assertions())
    daily_models.append((talks, daily_review))
    for presenter, row in zip(talks, daily_review):
        for reviewer, variable in zip(GROUPS, row):
            review_for[reviewer, presenter] = variable

for a, b in combinations(GROUPS, 2):
    joint_solver.add(Or(Not(review_for[a, b]), Not(review_for[b, a])))

status = joint_solver.check()
print("Z3 result with the no-reciprocal-review rule:", status)
assert status == sat

joint_model = joint_solver.model()
extended_z3_schedules = {
    day: read_assignments(joint_model, daily_review, talks, GROUPS)
    for day, (talks, daily_review) in enumerate(daily_models, start=1)
}

Z3 result with the no-reciprocal-review rule: sat


### Compare the complete schedules

Each matrix has 21 talk rows, grouped by day, and 21 reviewer columns. A 1 means the column's group reviews the talk in that row. Each row has six 1s, and each column has two 1s per day.

Because each group has one talk, every pair of groups corresponds to two cells on opposite sides of the main diagonal. The new rule says that those cells cannot both contain a 1. The code checks all pairs in each completed schedule.

In [11]:
def print_extended_matrix(method, schedules):
    print(f"\n{method}")
    print(f"{'Day':<5}{'Talk':<9}" + "".join(f"{i:>3}" for i in range(1, len(GROUPS) + 1)))
    for day, schedule in schedules.items():
        for presenter, reviewers in schedule.items():
            values = [int(group in reviewers) for group in GROUPS]
            print(f"{day:<5}{presenter:<9}" + "".join(f"{value:>3}" for value in values))

    reviews = {
        (reviewer, presenter)
        for schedule in schedules.values()
        for presenter, reviewers in schedule.items()
        for reviewer in reviewers
    }
    reciprocal_pairs = [
        (a, b) for a, b in combinations(GROUPS, 2)
        if (a, b) in reviews and (b, a) in reviews
    ]
    print("Reciprocal pairs:", len(reciprocal_pairs))
    assert not reciprocal_pairs


print_extended_matrix("Backtracking", extended_backtracking_schedules)
print_extended_matrix("Z3", extended_z3_schedules)


Backtracking
Day  Talk       1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21
1    Group1     0  0  1  1  0  1  0  0  0  1  0  0  0  1  0  0  1  0  0  0  0
1    Group2     0  0  0  0  1  0  1  0  1  0  0  1  0  0  1  0  0  0  0  1  0
1    Group3     0  0  0  0  0  1  0  1  0  0  1  0  1  0  0  1  0  0  1  0  0
1    Group4     0  1  0  0  0  0  1  0  1  0  0  1  0  0  0  0  0  1  0  0  1
1    Group5     1  0  1  0  0  0  0  1  0  0  0  0  1  0  0  1  0  0  1  0  0
1    Group6     0  1  0  1  0  0  0  0  0  1  0  0  0  1  0  0  0  1  0  1  0
1    Group7     1  0  0  0  1  0  0  0  0  0  1  0  0  0  1  0  1  0  0  0  1
2    Group8     0  0  0  1  0  1  0  0  0  1  1  0  1  0  0  1  0  0  0  0  0
2    Group9     1  0  0  0  1  0  0  0  0  0  0  1  0  1  1  0  0  0  0  1  0
2    Group10    0  1  1  0  0  0  1  0  0  0  0  0  1  0  0  0  1  1  0  0  0
2    Group11    1  0  0  0  1  0  0  0  1  0  0  0  0  1  0  0  0  0  1  0  1
2    Group12    0  0  1  0  0  1  0  1  0  1  0  0